In [ ]:
!pip install pycocotools opencv-python tqdm
!pip install ultralytics torch torchvision torchaudio --upgrade



In [ ]:
import json, os, cv2, random
from tqdm import tqdm

In [ ]:

# # ===== CONFIG =====
# PATCH_SIZE = 512         # patch dimension (square)
# OVERLAP = 0.3           # overlap between patches
# VAL_SPLIT = 0.10         # 10% patches for validation
# INPUT_IMG_DIR = "./train"
# ANNO_FILE = "./train/_annotations.coco.json"
# OUT_DIR = "patched_dataset"
# # ==================

# random.seed(42)

# # Create output dirs
# for sub in ["images/train", "labels/train", "images/val", "labels/val"]:
#     os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

# with open(ANNO_FILE, "r") as f:
#     coco = json.load(f)

# # Map image_id → filename & size
# id2img = {img["id"]: img for img in coco["images"]}

# # Group annotations by image
# annos_by_img = {}
# for ann in coco["annotations"]:
#     annos_by_img.setdefault(ann["image_id"], []).append(ann)

# all_patches = []  # (img, patch_id, patch, boxes)

# for img_id, img_info in tqdm(id2img.items(), desc="Creating patches"):
#     img_path = os.path.join(INPUT_IMG_DIR, img_info["file_name"])
#     image = cv2.imread(img_path)
#     if image is None:
#         print(f"⚠️ Could not read {img_path}")
#         continue

#     h, w = image.shape[:2]
#     step = int(PATCH_SIZE * (1 - OVERLAP))
#     patch_id = 0

#     for y0 in range(0, h, step):
#         for x0 in range(0, w, step):
#             x1, y1 = x0 + PATCH_SIZE, y0 + PATCH_SIZE
#             if x1 > w or y1 > h:
#                 continue
#             patch = image[y0:y1, x0:x1]
#             boxes = []
#             for ann in annos_by_img.get(img_id, []):
#                 bx, by, bw, bh = ann["bbox"]
#                 cx = bx + bw / 2
#                 cy = by + bh / 2
#                 if x0 <= cx < x1 and y0 <= cy < y1:
#                     new_x = (cx - x0) / PATCH_SIZE
#                     new_y = (cy - y0) / PATCH_SIZE
#                     new_w = bw / PATCH_SIZE
#                     new_h = bh / PATCH_SIZE
#                     if 0 < new_x < 1 and 0 < new_y < 1:
#                         boxes.append((0, new_x, new_y, new_w, new_h))  # class=0 (dot)

#             if boxes:
#                 base_name = f"{os.path.splitext(img_info['file_name'])[0]}_{patch_id}.jpg"
#                 all_patches.append((patch, boxes, base_name))
#                 patch_id += 1

# # ===== SPLIT INTO TRAIN / VAL =====
# random.shuffle(all_patches)
# n_val = int(len(all_patches) * VAL_SPLIT)
# val_patches = all_patches[:n_val]
# train_patches = all_patches[n_val:]

# def save_patches(subset, subset_name):
#     img_out = os.path.join(OUT_DIR, f"images/{subset_name}")
#     lbl_out = os.path.join(OUT_DIR, f"labels/{subset_name}")
#     for patch, boxes, name in tqdm(subset, desc=f"Saving {subset_name}"):
#         cv2.imwrite(os.path.join(img_out, name), patch)
#         with open(os.path.join(lbl_out, name.replace(".jpg", ".txt")), "w") as f:
#             for b in boxes:
#                 f.write(" ".join(map(str, b)) + "\n")

# save_patches(train_patches, "train")
# save_patches(val_patches, "val")

# print(f"\n✅ Done! Created {len(train_patches)} train and {len(val_patches)} val patches.")
# print(f"📁 Dataset saved to: {OUT_DIR}")

In [2]:
import yaml
import os

PATCHED_DIR = "./patched_dataset"  # your dataset path
DATA_YAML = os.path.join(PATCHED_DIR, "data.yaml")

data_dict = {
    "train": "images/train",
    "val": "images/val",
    "names": ["dot"]
}

# Write YAML
with open(DATA_YAML, "w") as f:
    yaml.dump(data_dict, f, default_flow_style=False)

print("✅ data.yaml created at:", DATA_YAML)


✅ data.yaml created at: ./patched_dataset\data.yaml


In [ ]:
from ultralytics import YOLO
import os

MODEL = "yolov8n.pt"
DATA_DICT = "./patched_dataset/data.yaml"
IMG_SIZE = 512
EPOCHS = 100
BATCH = 2
DEVICE = 0  # GPU

# ===== INITIALIZE MODEL =====
model = YOLO(MODEL)

# ===== TRAIN YOLOv8 =====
results = model.train(
    data=DATA_DICT,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=DEVICE,
    save=True,        # saves last.pt and best.pt
    save_period=5,     # saves checkpoint every 5 epochs
    amp =True
)

print("✅ Training complete!")
print("Best model saved at:", results.best)

New https://pypi.org/project/ultralytics/8.4.7 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.10.0 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./patched_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, nbs=6

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)


In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [ ]:
import torch
import platform

print("PyTorch version:", torch.__version__)
print("CUDA version bundled:", torch.version.cuda)
print("Python:", platform.python_version())

print("CUDA available? ", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("❌ CUDA still not visible to PyT")
